## Fase de Inicialização e Treinamento de modelos Causais e Seq2Seq que irão servir para o uso da **FastAPI**

# Primeiro, comece instalando os arquivos do `requirements.txt` e em seguida rode o script da próxima cédula, para instalar a biblioteca `torchao`

# `Esse procedimento será essencial para a implementação da API`

# Ao longo desse processo também estará realizado os tratamentos dos modelos de acordo como devem ser treinados esses

### LEVANTAMENTO DE MÉTRICAS POR MEIO DE GRÁFICOS DE CONVERGÊNCIA - Passo importante para saber como os modelos estão sendo treinados de acordo com os parâmetros de treinamento

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -U torchao

## Configuração da LoRa para Seq2Seq, seguindo mais abaixo na próxima cédula de código

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

# Definição do primeiro modelo a ser treinado
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# 2. Configurar o LoRA para Seq2Seq
# r=8 garante que a adaptação seja feita nos pesos de atenção (q, v)
# mantendo o custo computacional baixo como pede o edital
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"], # Módulos de atenção do T5
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Preparar o tokenizer para o formato de instrução do T5
def preprocess_function(examples):
    inputs = [f"Instruction: {instr} Input: {inp}" for instr, inp in zip(examples["instruction"], examples["input"])]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Setup do target (output)
    labels = tokenizer(examples["output"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Carregar dataset
from datasets import load_dataset
dataset = load_dataset("json", data_files="./dataset/dataset_gerado_curado.jsonl", split="train")
tokenized_dataset = dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


## O treinamento seguiu a ideia de dois modelos `Causais` e dois modelos `Seq2Seq` 

## Sendo esses os que irei abordar mais abaixo

## `Flat T5 Small - Seq2Seq`
## `sshleifer/bart-tiny-random - Seq2Seq`
## `Qwen/Qwen2-0.5B - Causal`
## `EleutherAI/gpt-neo-125M - Causal`

## A partir disso, faço uma análise e ``justificativa dos hiperparâmetros`` necessários para ``criação do modelo``

### Comparativo de Modelos e Hiperparâmetros

| Modelo | Arquitetura | VRAM Estimada | Tempo de Treino (aprox.) | Objetivo |
| :--- | :--- | :--- | :--- | :--- |
| **Flan-T5-Small** | Seq2Seq | Baixa | Rápido | Instrução simples |
| **Bart-Tiny** | Seq2Seq | Mínima | Muito Rápido | Baseline leve |
| **Qwen2-0.5B** | Causal | Média | Moderado | Raciocínio |
| **GPT-Neo-125M** | Causal | Média | Moderado | Text Gen |

In [ ]:
import torch
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

# 1. Limpeza de ambiente para evitar MemoryError
gc.collect()
torch.cuda.empty_cache()

# 2. Configuração do Modelo
model_id = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# 3. Configuração LoRA (Justificativa: r=8 p/ eficiência em hardware local)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)

# 4. Preparação do Dataset
def preprocess_function(examples):
    inputs = [f"Instruction: {instr} Input: {inp}" for instr, inp in zip(examples["instruction"], examples["input"])]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(examples["output"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Carrega o dataset da Etapa 1
dataset = load_dataset("json", data_files="./dataset/dataset_gerado_curado", split="train")
tokenized_dataset = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 5. Argumentos de Treino (Otimizados para evitar MemoryError)
training_args = Seq2SeqTrainingArguments(
    output_dir="./resultados_flan_t5",
    per_device_train_batch_size=1,      # Mínimo para economizar RAM
    gradient_accumulation_steps=8,     # Mantém o gradiente estável
    num_train_epochs=3,                # Ciclo completo de aprendizado
    learning_rate=2e-4,
    save_strategy="no",                # Evita sobrecarga de escrita em disco
    logging_steps=10,
    predict_with_generate=False,       # Evita inferência durante o treino
    report_to="none",
    dataloader_pin_memory=False,       # Necessário para CPU local
    optim="adafactor"                  # Otimizador específico p/ T5 (leve)
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# 6. Execução e Salvamento
print("Iniciando treinamento...")
trainer.train()

print("Treinamento concluído. Salvando modelo...")
model.save_pretrained("./models/lora_seq2seq_model_1")
tokenizer.save_pretrained("./models/lora_seq2seq_model_1")
print("✅ Modelo salvo com sucesso em ./lora_seq2seq_model_1")

``Observação: Optei por salvar todas as imagens na pasta images para uma melhor organização. Haja vista que o processador do computador estava lento e levando muito tempo, acabei optando por não salvar os modelos automaticamente por cédula de código após o treinamento rodar e tive que optar pelo colab, porém quando trouxe para dentro do VS code, as images no VS code sumiram, então eu baixei direto do Colab e assim foi possível trazer para esse projeto as seguintes imagens que estão salvas na pasta images. Foi um trabalho árduo, mas eu consegui com muita garra, desenvolver``




`Mais uma observação: No documento de relatório, deixo claro que usei os treinos de 10 em 10 steps, ao qual eu gravo numa tabela esses dados e faço a análise e crio o gráfico de convergência de alguns modelos como o da próximo cédula, deixando muito mais profissional, e em modelos menores eu fiz um desenvolvimento que já integrava essa funcionalidade de forma direta. 
`

## Na próxima etapa/cédula é possível, ao leitor visualizar a variavel low_raw, que fiz justamente esse passo, peguei toda a saída do comando passado do treino, e com isso fiz um gráfico, cujo gráfico está localizado na pasta image/ e nome: grafico_Flat_T5_Small.png

In [ ]:
import matplotlib.pyplot as plt
import re

# Cole aqui o texto exatamente como ele aparece na sua saída
log_raw = """
10 320.0468
20 296.7950
30 267.1584
40 230.1629
50 187.8319
60 140.4285
70 93.5745
80 69.3460
90 55.1912
100 47.5071
110 41.2349
120 41.4345
130 40.0919
140 39.1752
150 38.6014
160 37.8634
170 37.5924
180 37.0678
190 36.8269
200 36.5321
210 36.1029
220 34.1297
230 35.6387
240 35.7006
250 35.3540
260 35.1391
270 35.1491
280 34.9026
290 34.6342
300 34.5554
310 34.5088
320 34.3945
330 32.7475
"""

# Função para converter o texto em listas
def processar_log(texto):
    steps, losses = [], []
    for linha in texto.strip().split('\n'):
        s, l = linha.split()
        steps.append(int(s))
        losses.append(float(l))
    return steps, losses

steps, losses = processar_log(log_raw)

# Plotando
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker='o', label='Flat-T5-Small')
plt.title('Gráfico de Convergência')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

`O mesmo das cédulas passadas foi feito para os próximos modelos que aqui eu frizei, então o processo continuou sendo maçante, mas mesmo assim eu não desisti, tive umas decepções como por exemplo rodar por um bom período de tempo e no final por conta do processamento baixo do meu notebook não conseguir rodar algumas cédulas importantes, mas independente disso tudo, eu não parei e consegui fazer todo esse trajeto.`

In [ ]:
print(dataset.column_names)

In [ ]:
import torch
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

# 1. Limpeza
gc.collect()
torch.cuda.empty_cache()

# 2. Configuração (Modelo Tiny)
model_id = "sshleifer/bart-tiny-random"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# 3. Configuração LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)

# 4. Preparação do Dataset (AJUSTADA PARA BART)
def preprocess_function(examples):
    # Concatenar para o BART
    inputs = [f"Instruction: {instr} Input: {inp}" for instr, inp in zip(examples["instruction"], examples["input"])]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # BART usa 'text_target' em vez de as_target_tokenizer
    labels = tokenizer(text_target=examples["output"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset = load_dataset("json", data_files="./dataset/dataset_gerado_curado.jsonl", split="train")
tokenized_dataset = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 5. Argumentos de Treino
training_args = Seq2SeqTrainingArguments(
    output_dir="./resultados_tiny_bart",
    per_device_train_batch_size=2, # O tiny permite um batch um pouco maior
    gradient_accumulation_steps=4,
    num_train_epochs=5,           # Como é minúsculo, pode treinar por mais épocas
    learning_rate=3e-4,           # LR um pouco maior para modelos minúsculos
    save_strategy="no",
    logging_steps=5,
    report_to="none",
    optim="adafactor"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# 6. Execução
print("Iniciando treinamento do Tiny-BART...")
trainer.train()
model.save_pretrained("./dataset/lora_tiny_bart_final")
print("✅ Tiny-BART treinado e salvo!")

In [ ]:
import matplotlib.pyplot as plt
import re

# Cole aqui o texto exatamente como ele aparece na sua saída
log_raw = """
5	43.321216
10	43.321484
15	43.321130
20	43.321643
25	43.320599
30	43.321152
35	43.321027
40	43.320706
45	43.320895
50	43.321323
55	43.320120
60	43.321277
65	43.319885
70	43.319598
75	43.320013
80	43.319406
85	43.318848
90	43.319315
95	43.318756
100	43.318289
105	43.318552
110	38.986594
115	43.318185
120	43.317746
125	43.317737
130	43.317828
135	43.317123
140	43.317633
145	43.316690
150	43.316742
155	43.316690
160	43.316223
165	43.315649
170	43.316302
175	43.315692
180	43.316684
185	43.315234
190	43.313983
195	43.314633
200	43.313361
205	43.313940
210	43.313763
215	43.313162
220	38.982047
225	43.312561
230	43.313126
235	43.311749
240	43.311810
245	43.311783
250	43.312109
255	43.311349
260	43.311008
265	43.311176
270	43.310498
275	43.311029
280	43.310327
285	43.309619
290	43.310236
295	43.309879
300	43.308783
305	43.308801
310	43.308331
315	43.308789
320	43.308679
325	43.307312
330	38.977225
335	43.307251
340	43.307834
345	43.307349
350	43.306607
355	43.306506
360	43.306128
365	43.306418
370	43.306049
375	43.305792
380	43.305121
385	43.304916
390	43.305569
395	43.304910
400	43.304956
405	43.304333
410	43.304559
415	43.304266
420	43.304007
425	43.304028
430	43.303857
435	43.303946
440	38.973169
445	43.303455
450	43.303485
455	43.303128
460	43.303848
465	43.303256
470	43.303348
475	43.303223
480	43.302954
485	43.302399
490	43.302576
495	43.302612
500	43.303162
505	43.302594
510	43.302353
515	43.302457
520	43.302240
525	43.302332
530	43.302029
535	43.301917
540	43.302188
545	43.302756
550	38.972260
"""

# Função para converter o texto em listas
def processar_log(texto):
    steps, losses = [], []
    for linha in texto.strip().split('\n'):
        s, l = linha.split()
        steps.append(int(s))
        losses.append(float(l))
    return steps, losses

steps, losses = processar_log(log_raw)

# Plotando
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker='o', label='bart-tiny-random')
plt.title('Gráfico de Convergência')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
import matplotlib.pyplot as plt # Import para o gráfico

# 1. Configuração do Modelo
model_id = "Qwen/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

# 2. Configuração LoRA
config = LoraConfig(
    r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)

# 3. Dataset
dataset = load_dataset("json", data_files="./dataset/dataset_gerado_curado.jsonl", split="train")

def preprocess(examples):
    text = [f"User: {instr} {inp} Assistant: {out}"
            for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"])]
    model_inputs = tokenizer(text, max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

tokenized = dataset.map(preprocess, batched=True)

# 4. Treino com Log ativado
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./resultados_treino",
        per_device_train_batch_size=1,
        max_steps=100,
        learning_rate=2e-5,
        save_strategy="no",
        report_to="none",
        use_cpu=True,
        logging_steps=10,        # <--- ESSENCIAL: Loga a cada 10 passos
        logging_strategy="steps" # <--- ESSENCIAL: Define a estratégia de log
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("Iniciando treino local na CPU...")
trainer.train()

# 5. Extração e Gráfico (A Mágica acontece aqui!)
# Pega o histórico de logs do objeto trainer
logs = trainer.state.log_history
# Filtra apenas os logs que possuem a loss (ignora outros logs de sistema)
loss_values = [log['loss'] for log in logs if 'loss' in log]
steps = [log['step'] for log in logs if 'loss' in log]

print("Treino concluído! Gerando gráfico...")

plt.figure(figsize=(10, 5))
plt.plot(steps, loss_values, marker='o', linestyle='-', color='b', label='Qwen2-0.5B')
plt.title('Gráfico de Convergência - Fine-Tuning LoRA')
plt.xlabel('Steps')
plt.ylabel('Training Loss')
plt.grid(True)
plt.legend()
plt.show()

# 6. Salvar
model.save_pretrained("./dataset/modelo_final_lora")
print("Pasta './dataset/modelo_final_lora' gerada com sucesso.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import matplotlib.pyplot as plt
import os

# 1. Configuração do Modelo
model_id = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Carregando modelo...")
model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Configuração LoRA
model = get_peft_model(model, LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
))

# 3. Processamento do Dataset
dataset = load_dataset("json", data_files="./dataset/dataset_gerado_curado.jsonl", split="train")

def preprocess(ex):
    text = [f"Pergunta: {i} Resposta: {o}" for i, o in zip(ex['instruction'], ex['output'])]
    model_inputs = tokenizer(text, max_length=64, truncation=True, padding="max_length")
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)

# 4. Treino com Log ativado (logging_steps=10)
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./modelo_final",
        max_steps=100,            # Aumentado para podermos ver a curva
        per_device_train_batch_size=1,
        use_cpu=True,
        save_strategy="no",
        logging_steps=10,         # Essencial para gerar os dados do gráfico
        report_to="none"
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("Iniciando Treino...")
trainer.train()

# 5. Extração e Plotagem (Solução Definitiva)
logs = trainer.state.log_history
steps = [log['step'] for log in logs if 'loss' in log]
losses = [log['loss'] for log in logs if 'loss' in log]

# Cria a pasta 'images' se não existir
os.makedirs("images", exist_ok=True)

# Gera o gráfico
plt.figure(figsize=(10, 6))
plt.plot(steps, losses, marker='o', linestyle='-', color='red', label='GPT-Neo Loss')
plt.title('Convergência de Treinamento - GPT-Neo')
plt.xlabel('Steps')
plt.ylabel('Training Loss')
plt.grid(True, linestyle='--')
plt.legend()

# Salva na pasta images
plt.savefig("images/grafico_gpt_neo.png", dpi=300, bbox_inches='tight')
print("Treino concluído e gráfico salvo em 'images/grafico_gpt_neo.png'")

# 6. Salvar Modelo
model.save_pretrained("./dataset/modelo_final")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def carregar_modelo(model_key, registry):
    # Pega as infos do registro
    base_model_id, lora_path, is_seq2seq = registry[model_key]

    # Carrega o Tokenizer e o Modelo Base
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)

    if is_seq2seq:
        base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_id)
    else:
        base_model = AutoModelForCausalLM.from_pretrained(base_model_id)

    # Carrega o adaptador LoRA
    model = PeftModel.from_pretrained(base_model, lora_path)

    return model, tokenizer